<a href="https://colab.research.google.com/github/Orti-G/Thesis/blob/ML_Training/ThesisAnomaly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import os

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble       import IsolationForest
from sklearn.svm            import OneClassSVM
from sklearn.neighbors      import LocalOutlierFactor
from sklearn.metrics        import (
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
TRAIN_CSV = 'synthetic_household_training_data.csv'
TEST_CSV = 'synthetic_household_test_session_more_anomalies.csv'
MODEL_OUT = 'anomaly_model.pkl'

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_train['Timestamp'] = pd.to_datetime(df_train['Timestamp'], dayfirst=True)
df_train = df_train.sort_values('Timestamp').reset_index(drop=True)

# Drop Note if present — verify all rows are normal first
if 'Note' in df_train.columns:
    non_normal = (df_train['Note'] != 'normal').sum()
    if non_normal > 0:
        print(f"⚠️  {non_normal} non-normal rows found in training CSV — dropping them.")
        df_train = df_train[df_train['Note'] == 'normal'].copy()
    df_train = df_train.drop(columns=['Note'])

print(f"Training shape : {df_train.shape}")
print(f"Date range     : {df_train['Timestamp'].min()}  →  {df_train['Timestamp'].max()}")
print(f"\n{df_train.head(3).to_string()}")

Training shape : (6048, 13)
Date range     : 2025-01-06 00:00:00  →  2025-01-26 23:55:00

            Timestamp  Voltage (V)  Current (A)  Power (W)  Interval kWh  Hour  Day  Weekend  Roll mean 1hr  Roll mean 24hr  Dev 1hr  Dev 24hr  Cumul kWh
0 2025-01-06 00:00:00        219.6         2.53      554.8        0.0462     0    1        0          554.8           554.8      0.0       0.0      0.046
1 2025-01-06 00:05:00        221.0         2.00      442.2        0.0368     0    1        0          498.5           498.5    -56.3     -56.3      0.083
2 2025-01-06 00:10:00        222.3         5.92     1315.5        0.1096     0    1        0          770.8           770.8    544.7     544.7      0.193


In [ ]:
df_test = pd.read_csv(TEST_CSV)
df_test['Timestamp'] = pd.to_datetime(df_test['Timestamp'], dayfirst=True)
df_test = df_test.sort_values('Timestamp').reset_index(drop=True)

assert 'Note' in df_test.columns, "❌ Test CSV must have a Note column for evaluation"

# Isolate labels NOW — never touch them again until evaluation
test_labels_raw = df_test['Note'].copy()          # 'normal', 'phone_charger', etc.
y_true = (test_labels_raw != 'normal').astype(int) # 1 = anomaly, 0 = normal

print(f"Test shape  : {df_test.shape}")
print(f"Date range  : {df_test['Timestamp'].min()}  →  {df_test['Timestamp'].max()}")
print(f"\nLabel breakdown:")
print(test_labels_raw.value_counts().to_string())
print(f"\n  Normal  : {(y_true == 0).sum()}")
print(f"  Anomaly : {(y_true == 1).sum()}")

Test shape  : (1000, 14)
Date range  : 2025-06-01 00:00:00  →  2025-09-01 11:15:00

Label breakdown:
Note
normal            960
sudden_drop        13
phone_charger      11
laptop_charger      8
power_spike         8

  Normal  : 960
  Anomaly : 40


In [ ]:
ANOMALY_FEATURES = [
    'Voltage (V)',
    'Current (A)',
    'Power (W)',
    'Interval kWh',
    'Roll mean 1hr',
    'Roll mean 24hr',
    'Dev 1hr',
    'Dev 24hr',
]

# Sanity check — every feature must exist in both CSVs
for col in ANOMALY_FEATURES:
    assert col in df_train.columns, f"❌ Missing from train: {col}"
    assert col in df_test.columns,  f"❌ Missing from test : {col}"

X_train_raw = df_train[ANOMALY_FEATURES].values
X_test_raw  = df_test[ANOMALY_FEATURES].values

print(f"X_train shape : {X_train_raw.shape}")
print(f"X_test  shape : {X_test_raw.shape}")
print(f"\nTraining feature stats (low variance is by design):")
print(df_train[ANOMALY_FEATURES].describe().round(3).to_string())

X_train shape : (6048, 8)
X_test  shape : (1000, 8)

Training feature stats (low variance is by design):
       Voltage (V)  Current (A)  Power (W)  Interval kWh  Roll mean 1hr  Roll mean 24hr   Dev 1hr  Dev 24hr
count     6048.000     6048.000   6048.000      6048.000       6048.000        6048.000  6048.000  6048.000
mean       220.017        4.968   1092.864         0.091       1092.678        1080.490     0.186    12.374
std          2.990        1.786    392.357         0.033        367.342          66.923   162.996   391.742
min        209.600        1.680    380.000         0.032        491.900         498.500  -936.300  -720.800
25%        218.000        3.700    812.450         0.068        816.125        1087.900   -63.925  -265.650
50%        220.000        4.955   1090.050         0.091       1125.000        1090.900    -6.200     6.550
75%        222.000        5.790   1272.150         0.106       1238.150        1095.700    56.725   212.725
max        233.400       10.600

In [ ]:
# Fit ONLY on training data — test set is unseen
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)      # transform only, no refit

print("StandardScaler fit on training data only.")
print(f"\nLearned means : {dict(zip(ANOMALY_FEATURES, scaler.mean_.round(3)))}")
print(f"\nPost-scale train mean ≈ 0 : {X_train.mean(axis=0).round(3)}")
print(f"Post-scale train std  ≈ 1 : {X_train.std(axis=0).round(3)}")

StandardScaler fit on training data only.

Learned means : {'Voltage (V)': np.float64(220.017), 'Current (A)': np.float64(4.968), 'Power (W)': np.float64(1092.864), 'Interval kWh': np.float64(0.091), 'Roll mean 1hr': np.float64(1092.678), 'Roll mean 24hr': np.float64(1080.49), 'Dev 1hr': np.float64(0.186), 'Dev 24hr': np.float64(12.374)}

Post-scale train mean ≈ 0 : [-0. -0.  0.  0.  0.  0. -0. -0.]
Post-scale train std  ≈ 1 : [1. 1. 1. 1. 1. 1. 1. 1.]


In [ ]:
# Isolation Forest
#   contamination='auto' → decision threshold at anomaly score = 0
#   (correct when training set is all-normal, no contamination to estimate)
iso_forest = IsolationForest(
    n_estimators  = 200,
    contamination = 0.01,
    max_features  = 1.0,
    random_state  = RANDOM_STATE,
    n_jobs        = -1,
)
iso_forest.fit(X_train)
print("✅ Isolation Forest trained")

# One-Class SVM
#   nu=0.01 → upper bound on fraction of training errors (very tight)
#   gamma='scale' → 1 / (n_features * X.var())
oc_svm = OneClassSVM(
    kernel = 'rbf',
    nu     = 0.01,
    gamma  = 'scale',
)
oc_svm.fit(X_train)
print("✅ One-Class SVM trained")

# Local Outlier Factor
#   novelty=True is REQUIRED — enables predict() on new/unseen data
#   contamination=0.01 → tight boundary, assumes very few outliers in neighbourhood
lof = LocalOutlierFactor(
    n_neighbors   = 20,
    novelty       = True,
    contamination = 0.01,
    n_jobs        = -1,
)
lof.fit(X_train)
print("✅ Local Outlier Factor trained")

✅ Isolation Forest trained
✅ One-Class SVM trained
✅ Local Outlier Factor trained


In [ ]:
# All models output: +1 = normal, -1 = anomaly
pred_if   = iso_forest.predict(X_test)
pred_svm  = oc_svm.predict(X_test)
pred_lof  = lof.predict(X_test)

predictions = {
    'Isolation Forest'    : pred_if,
    'One-Class SVM'       : pred_svm,
    'Local Outlier Factor': pred_lof,
}

print("Raw prediction counts (+1 normal / -1 anomaly):\n")
for name, preds in predictions.items():
    n_normal  = (preds ==  1).sum()
    n_anomaly = (preds == -1).sum()
    print(f"  {name:<25}  normal={n_normal}  flagged={n_anomaly}")

Raw prediction counts (+1 normal / -1 anomaly):

  Isolation Forest           normal=921  flagged=79
  One-Class SVM              normal=887  flagged=113
  Local Outlier Factor       normal=824  flagged=176


In [ ]:
results = []

for name, preds in predictions.items():
    y_pred = (preds == -1).astype(int)   # convert: -1 → 1 (anomaly), +1 → 0 (normal)

    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true,    y_pred, zero_division=0)
    f1   = f1_score(y_true,        y_pred, zero_division=0)

    cm              = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp  = cm.ravel()

    results.append({
        'Model'    : name,
        'TP'       : int(tp),
        'FP'       : int(fp),
        'FN'       : int(fn),
        'TN'       : int(tn),
        'Precision': round(prec, 4),
        'Recall'   : round(rec,  4),
        'F1 Score' : round(f1,   4),
    })

    print(f"\n── {name} ──")
    print(classification_report(y_true, y_pred,
                                 target_names=['Normal', 'Anomaly'],
                                 zero_division=0))

df_results = (pd.DataFrame(results)
                .sort_values(['F1 Score', 'FP'], ascending=[False, True])
                .reset_index(drop=True))

print("\n" + "=" * 65)
print("SUMMARY TABLE")
print("=" * 65)
print(df_results.to_string(index=False))


── Isolation Forest ──
              precision    recall  f1-score   support

      Normal       0.99      0.95      0.97       960
     Anomaly       0.43      0.85      0.57        40

    accuracy                           0.95      1000
   macro avg       0.71      0.90      0.77      1000
weighted avg       0.97      0.95      0.96      1000


── One-Class SVM ──
              precision    recall  f1-score   support

      Normal       1.00      0.92      0.96       960
     Anomaly       0.35      1.00      0.52        40

    accuracy                           0.93      1000
   macro avg       0.68      0.96      0.74      1000
weighted avg       0.97      0.93      0.94      1000


── Local Outlier Factor ──
              precision    recall  f1-score   support

      Normal       1.00      0.86      0.92       960
     Anomaly       0.23      1.00      0.37        40

    accuracy                           0.86      1000
   macro avg       0.61      0.93      0.65      1000
w

In [ ]:
anomaly_types = sorted(test_labels_raw[test_labels_raw != 'normal'].unique())

print("PER-ANOMALY-TYPE DETECTION RATE\n" + "-" * 55)
for name, preds in predictions.items():
    y_pred_s = pd.Series((preds == -1).astype(int), index=df_test.index)
    print(f"\n  {name}")
    for atype in anomaly_types:
        mask   = test_labels_raw == atype
        caught = y_pred_s[mask].sum()
        total  = mask.sum()
        rate   = caught / total if total > 0 else 0
        bar    = '█' * int(rate * 20) + '░' * (20 - int(rate * 20))
        print(f"    {atype:<18}  {bar}  {caught}/{total}  ({rate:.0%})")

PER-ANOMALY-TYPE DETECTION RATE
-------------------------------------------------------

  Isolation Forest
    laptop_charger      █████████████████░░░  7/8  (88%)
    phone_charger       ████████████████░░░░  9/11  (82%)
    power_spike         ████████████████████  8/8  (100%)
    sudden_drop         ███████████████░░░░░  10/13  (77%)

  One-Class SVM
    laptop_charger      ████████████████████  8/8  (100%)
    phone_charger       ████████████████████  11/11  (100%)
    power_spike         ████████████████████  8/8  (100%)
    sudden_drop         ████████████████████  13/13  (100%)

  Local Outlier Factor
    laptop_charger      ████████████████████  8/8  (100%)
    phone_charger       ████████████████████  11/11  (100%)
    power_spike         ████████████████████  8/8  (100%)
    sudden_drop         ████████████████████  13/13  (100%)


In [ ]:
winner_row   = df_results.iloc[0]   # already sorted by F1 desc, FP asc
winner_name  = winner_row['Model']
winner_model = {'Isolation Forest': iso_forest,
                'One-Class SVM'   : oc_svm,
                'Local Outlier Factor': lof}[winner_name]

winner_metrics = {
    'precision': winner_row['Precision'],
    'recall'   : winner_row['Recall'],
    'f1'       : winner_row['F1 Score'],
}

print("=" * 45)
print(f"  🏆  WINNER : {winner_name}")
print(f"      Precision : {winner_metrics['precision']:.4f}")
print(f"      Recall    : {winner_metrics['recall']:.4f}")
print(f"      F1 Score  : {winner_metrics['f1']:.4f}")
print(f"      TP / FP   : {winner_row['TP']} / {winner_row['FP']}")
print("=" * 45)

  🏆  WINNER : Isolation Forest
      Precision : 0.4304
      Recall    : 0.8500
      F1 Score  : 0.5714
      TP / FP   : 34 / 45


In [ ]:
anomaly_bundle = {
    'model'       : winner_model,
    'scaler'      : scaler,
    'feature_cols': ANOMALY_FEATURES,
    'model_name'  : winner_name,
    'metrics'     : winner_metrics,
}

joblib.dump(anomaly_bundle, MODEL_OUT)

size_kb = os.path.getsize(MODEL_OUT) / 1024
print(f"✅ Saved → {MODEL_OUT}")
print(f"   Size    : {size_kb:.1f} KB")
print(f"   Keys    : {list(anomaly_bundle.keys())}")

✅ Saved → anomaly_model.pkl
   Size    : 2310.9 KB
   Keys    : ['model', 'scaler', 'feature_cols', 'model_name', 'metrics']


In [ ]:
loaded = joblib.load(MODEL_OUT)
print(f"Loaded model  : {loaded['model_name']}")
print(f"Feature cols  : {loaded['feature_cols']}")
print(f"Metrics       : {loaded['metrics']}\n")

# --- Normal reading ---
normal_reading = pd.DataFrame([{
    'Voltage (V)'   : 220.0,
    'Current (A)'   : 4.5,
    'Power (W)'     : 990.0,
    'Interval kWh'  : 0.0825,
    'Roll mean 1hr' : 1000.0,
    'Roll mean 24hr': 1020.0,
    'Dev 1hr'       : -10.0,
    'Dev 24hr'      : -30.0,
}])
X = loaded['scaler'].transform(normal_reading[loaded['feature_cols']])
out = loaded['model'].predict(X)[0]
print(f"Normal reading   → raw={out:+d}  →  {'✅ NORMAL' if out == 1 else '🚨 ANOMALY'}")

# --- Phone charger anomaly ---
phone_reading = pd.DataFrame([{
    'Voltage (V)'   : 219.2,
    'Current (A)'   : 0.052,
    'Power (W)'     : 11.2,
    'Interval kWh'  : 0.00093,
    'Roll mean 1hr' : 950.0,
    'Roll mean 24hr': 1010.0,
    'Dev 1hr'       : -938.8,
    'Dev 24hr'      : -998.8,
}])
X = loaded['scaler'].transform(phone_reading[loaded['feature_cols']])
out = loaded['model'].predict(X)[0]
print(f"Phone charger    → raw={out:+d}  →  {'✅ NORMAL' if out == 1 else '🚨 ANOMALY'}")

Loaded model  : Isolation Forest
Feature cols  : ['Voltage (V)', 'Current (A)', 'Power (W)', 'Interval kWh', 'Roll mean 1hr', 'Roll mean 24hr', 'Dev 1hr', 'Dev 24hr']
Metrics       : {'precision': np.float64(0.4304), 'recall': np.float64(0.85), 'f1': np.float64(0.5714)}

Normal reading   → raw=+1  →  ✅ NORMAL
Phone charger    → raw=-1  →  🚨 ANOMALY


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
